In [1]:
import os
import json
import pandas as pd

from pathlib import Path


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 89 samples and 17014 columns.
First few rows of the data:


,overall survival (os) (month),outcome,samples,age,Sex,t stage,n stage,ajcc/uicc-tnm stage,OR4F5,OR4F29,...,COL6A2,FTCD,LSS,MCM3AP,YBEY,C21orf58,PCNT,DIP2A,S100B,PRMT2
0,34,1,CRC_1,63,Male,T4b,N0,IV,2.238720,3.576052,...,6.904159,0.513126,5.505524,4.826873,2.024319,1.362392,4.871521,4.314822,1.041236,3.873577
1,34,0,CRC_2,72,Female,T3,N2a,IV,2.690912,4.288482,...,6.167773,1.052961,4.406301,4.852319,2.342852,1.545884,5.215702,4.589853,1.031734,3.731309
2,33,0,CRC_3,58,Male,T4a,N2a,IV,2.906905,3.063055,...,7.235745,0.971923,4.432446,4.652795,1.590173,0.714812,5.113489,4.286788,1.482064,3.758579
3,32,0,CRC_4,73,Male,T4a,N2b,IV,3.284267,3.798752,...,6.739188,1.031439,4.180149,4.711043,1.433220,1.109589,4.941454,4.435966,1.507448,3.652840
4,28,1,CRC_5,66,Female,T3,N2a,IV,4.136243,5.140208,...,5.951511,1.995989,4.102097,4.297774,1.687317,1.307901,4.931410,4.270836,1.811508,3.717703


In [2]:
n_stats_written = 0

for idx, endpoint in enumerate(summary_data['survival-endpoints']):
    time_var = endpoint['time_var']['var_name']
    event_var = endpoint['event_var']['var_name']

    if time_var not in df.columns or event_var not in df.columns:
        abbrv = endpoint.get('abbrv', f'endpoint[{idx}]')
        print(
            f"Skipping stats for {abbrv}: columns not in preprocessed table "
            f"({time_var!r}, {event_var!r})."
        )
        endpoint.pop('stats', None)
        continue

    complete_mask = df[[time_var, event_var]].notnull().all(axis=1)
    n_complete = int(complete_mask.sum())
    n_incomplete = int(len(df) - n_complete)

    # Among samples with complete time + event data
    censored_count = int((complete_mask & (df[event_var] == 0.0)).sum())
    event_count = int((complete_mask & (df[event_var] == 1.0)).sum())

    censored_ratio = censored_count / n_complete

    print(f"Number of samples with incomplete data for {time_var} or {event_var}: {n_incomplete}")
    print(f"Number of samples with complete data for both variables: {n_complete}")
    print(f"Number of censored samples (event_var == 0.0): {censored_count}")
    print(f"Number of events (event_var == 1.0): {event_count}")
    pct = 100.0 * censored_ratio
    print(
        f"Censored ratio (among complete): {censored_count} / {n_complete} = "
        f"{censored_ratio:.3f} ({pct:.1f}%)"
    )

    # Refresh only this block on each run; leave all other endpoint fields unchanged
    endpoint['stats'] = {
        'n_incomplete': n_incomplete,
        'n_complete': n_complete,
        'n_censored': censored_count,
        'n_events': event_count,
        'censored_ratio': round(censored_ratio, 2),
    }
    n_stats_written += 1

with open(description_path, 'w', encoding='utf-8') as f:
    json.dump(summary_data, f, indent=4, ensure_ascii=False)
    f.write('\n')

print(
    f"Updated stats for {n_stats_written} of {len(summary_data['survival-endpoints'])} endpoint(s); "
    f"wrote {description_path.resolve()}"
)


Number of samples with incomplete data for overall survival (os) (month) or outcome: 0
Number of samples with complete data for both variables: 89
Number of censored samples (event_var == 0.0): 17
Number of events (event_var == 1.0): 72
Censored ratio (among complete): 17 / 89 = 0.191 (19.1%)
Updated stats for 1 of 1 endpoint(s); wrote /Users/jakakokosar/dev/phd/survival-rnaseq-data/datasets/geo/GSE192667/description.json
